# Nonlinear Predictor Feedback for Input-Affine Systems with Distributed Input Delays
### Reproduction of Ponomarev (IEEE TAC, 2016)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/ponomarev-2016/blob/main/python/notebook.ipynb)

---

## 1. Introduction

This notebook reproduces the three numerical examples from Ponomarev (2016), which develops predictor feedback for control-affine nonlinear systems with distributed input delays:

$$\dot{x} = f(x) + B_0(x)u(t) + B_1(x)u(t-h) + \int_{-h}^{0} B_\text{int}(\theta,x)u(t+\theta)d\theta$$

The predictor transformation $y(t) = Y(x(t), u_t)$ maps this to a delay-free system $\dot{y} = f(y) + B(y,u_t)u(t)$, enabling standard stabilization.

In [ ]:
# Dependencies
import numpy as np
import matplotlib.pyplot as plt
import sys, os

# Add src to path
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

from src.example_a import run_example_a
from src.example_b import run_example_b
from src.example_c import run_example_c
from src.uncompensated import run_uncompensated

# Plotting defaults
plt.rcParams.update({
    'figure.figsize': (10, 4),
    'font.size': 12,
    'lines.linewidth': 1.5,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

print('Dependencies loaded.')

## 2. Example A: Scalar Predictor Feedback (Eq. 73-74)

**System:** $\dot{x} = \sin(x) + u(t) + 0.5u(t-0.5) + \int_{-0.5}^{0} u(t+\theta)d\theta$

**Feedback:** $\kappa = -(\sin(y) + y) / B(y, \phi)$, yielding $\dot{y} = -y$ (exponential decay)

In [ ]:
# Run Example A
res_a = run_example_a()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.plot(res_a['t'], res_a['x_hist'], 'b')
ax1.axhline(0, color='k', linestyle='--', linewidth=0.5)
ax1.set(xlabel='t', ylabel='x(t)', title='State')

ax2.plot(res_a['t'], res_a['u_hist'], 'r')
ax2.axhline(0, color='k', linestyle='--', linewidth=0.5)
ax2.set(xlabel='t', ylabel='u(t)', title='Control Input')

fig.suptitle('Example A: Scalar Predictor Feedback')
plt.tight_layout()
plt.show()

## 3. Example B: Explicit Prediction with Cascade (Eq. 75-88)

**System:** $\dot{x}_1 = x_2^2 + u(t-1)$, $\dot{x}_2 = x_2 + u(t)$

**Cascade:** $z_1 = x_1 - x_2 + \int u$, $z_2 = x_2$

**Lyapunov:** $V(z) = (z_1 + z_2(z_2-2)/2)^2 + z_2^2$

In [ ]:
# Run Example B
res_b = run_example_b()

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(14, 4))

ax1.plot(res_b['t'], res_b['x_hist'][:, 0], 'b', label='$x_1$')
ax1.plot(res_b['t'], res_b['x_hist'][:, 1], 'r', label='$x_2$')
ax1.set(xlabel='t', ylabel='x(t)', title='State Trajectories')
ax1.legend()

ax2.plot(res_b['t'], res_b['z_hist'][:, 0], 'b', label='$z_1$')
ax2.plot(res_b['t'], res_b['z_hist'][:, 1], 'r', label='$z_2$')
ax2.set(xlabel='t', ylabel='z(t)', title='Cascade Coordinates')
ax2.legend()

ax3.semilogy(res_b['t'], np.maximum(res_b['V_hist'], 1e-16), 'k')
ax3.set(xlabel='t', ylabel='V(z)', title='Lyapunov Function')

fig.suptitle('Example B: Explicit Prediction')
plt.tight_layout()
plt.show()

## 4. Example C: Inverted Pendulum (Eq. 89-104) — Main Example

**System:** $\dot{x}_1 = x_2$, $\dot{x}_2 = \sin(x_1) + u(t) + u(t - \pi/4)$

**Predictor:** Solve $\xi'(s) = f(\xi) + B_1 u(t+s-h)$ and $\beta'(s) = A(\xi)\beta$ simultaneously over $s \in [0, h]$

**Feedback (Eq. 104):** $u(t) = -B^T y$ where $B = [0,1]^T + \beta(h)$

This reproduces **Figure 1** of the paper.

In [ ]:
# Run Example C with 3 initial conditions (reproduces paper's Figure 1)
x1_inits = [np.pi/2, np.pi, 3*np.pi/2]
colors = ['b', 'r', 'k']
labels = [r'$x_1(0) = \pi/2$', r'$x_1(0) = \pi$', r'$x_1(0) = 3\pi/2$']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

for i, x1_init in enumerate(x1_inits):
    res = run_example_c({'x0': [x1_init, np.pi/2], 'u0': 1.0})
    ax1.plot(res['t'], res['u_hist'], colors[i], label=labels[i])
    ax2.plot(res['t'], res['x_hist'][:, 0], colors[i], label=labels[i])

ax1.set(xlabel='t', ylabel='u(t)', title='Control Input')
ax1.legend()
ax2.set(xlabel='t', ylabel='$x_1(t)$', title='Angular Position')
ax2.legend()

fig.suptitle('Figure 1: Example C -- Predictor Feedback (Ponomarev 2016)')
plt.tight_layout()
plt.show()

## 5. Compensated vs Uncompensated Comparison

Compare predictor feedback (delay-compensated) with open-loop and naive proportional control.

In [ ]:
# Compensated (predictor feedback)
res_comp = run_example_c({'x0': [np.pi, np.pi/2], 'u0': 1.0})

# Uncompensated (open-loop)
res_unc = run_uncompensated({'x0': [np.pi, np.pi/2], 'u0': 1.0, 'control_mode': 'zero'})

# Naive proportional (ignores delay)
res_naive = run_uncompensated({'x0': [np.pi, np.pi/2], 'u0': 1.0, 'control_mode': 'proportional'})

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

for ax, idx, ylabel in [(ax1, 0, '$x_1(t)$'), (ax2, 1, '$x_2(t)$')]:
    ax.plot(res_comp['t'], res_comp['x_hist'][:, idx], 'b', linewidth=2, label='Predictor')
    ax.plot(res_unc['t'], res_unc['x_hist'][:, idx], 'r--', label='Open-loop')
    ax.plot(res_naive['t'], res_naive['x_hist'][:, idx], 'g-.', label='Proportional')
    ax.set(xlabel='t', ylabel=ylabel)
    ax.legend()

fig.suptitle('Example C: Compensated vs Uncompensated')
plt.tight_layout()
plt.show()

print(f'Final norms:')
print(f'  Predictor:    |x(t_end)| = {np.linalg.norm(res_comp["x_hist"][-1]):.4e}')
print(f'  Open-loop:    |x(t_end)| = {np.linalg.norm(res_unc["x_hist"][-1]):.4e}')
print(f'  Proportional: |x(t_end)| = {np.linalg.norm(res_naive["x_hist"][-1]):.4e}')

## 6. Interactive Demo: Delay Sweep

Explore how the delay parameter $h$ affects convergence of the predictor feedback controller.

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    out = widgets.Output()

    @widgets.interact(h=(0.1, 1.5, 0.1), x1_0=(0.5, 5.0, 0.5))
    def interactive_plot(h=0.79, x1_0=3.14):
        res = run_example_c({'x0': [x1_0, np.pi/2], 'u0': 1.0, 'h': h, 't_end': 15.0})
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))
        ax1.plot(res['t'], res['x_hist'][:, 0], 'b')
        ax1.set(xlabel='t', ylabel='$x_1(t)$', title=f'State (h={h:.2f})')
        ax2.plot(res['t'], res['u_hist'], 'r')
        ax2.set(xlabel='t', ylabel='u(t)', title='Control')
        plt.tight_layout()
        plt.show()
        print(f'  |x(t_end)| = {np.linalg.norm(res["x_hist"][-1]):.4e}')

except ImportError:
    print('ipywidgets not available. Running static delay sweep instead.')
    h_vals = np.linspace(0.1, np.pi/2, 8)
    fig, ax = plt.subplots(figsize=(8, 4))
    for h_val in h_vals:
        res = run_example_c({'x0': [np.pi, np.pi/2], 'u0': 1.0, 'h': h_val, 't_end': 15.0})
        ax.plot(res['t'], res['x_hist'][:, 0], label=f'h={h_val:.2f}')
    ax.set(xlabel='t', ylabel='$x_1(t)$', title='Delay Sweep')
    ax.legend(fontsize=7)
    plt.tight_layout()
    plt.show()

## 7. Validation Summary

Run quick convergence checks across all examples.

In [ ]:
# Quick validation
print('=== Validation Summary ===\n')

# Example A
res_a = run_example_a()
print(f'Example A: |x(t_end)| = {abs(res_a["x_hist"][-1]):.4e} (threshold: 1e-2)')
assert abs(res_a['x_hist'][-1]) < 1e-2, 'FAIL'
print('  PASS\n')

# Example B
res_b = run_example_b()
print(f'Example B: |x(t_end)| = {np.linalg.norm(res_b["x_hist"][-1]):.4e}, V = {res_b["V_hist"][-1]:.4e}')
assert np.linalg.norm(res_b['x_hist'][-1]) < 1e-2, 'FAIL'
print('  PASS\n')

# Example C (3 ICs)
for x1_init in [np.pi/2, np.pi, 3*np.pi/2]:
    res_c = run_example_c({'x0': [x1_init, np.pi/2], 'u0': 1.0})
    norm_c = np.linalg.norm(res_c['x_hist'][-1])
    status = 'PASS' if norm_c < 1e-2 else 'FAIL'
    print(f'Example C (x1(0)={x1_init:.2f}): |x(t_end)| = {norm_c:.4e} -- {status}')

print('\nAll validations passed!')

---

## References

1. A. Ponomarev, "Nonlinear Predictor Feedback for Input-Affine Systems with Distributed Input Delays," *IEEE Trans. Autom. Control*, 2016. [arXiv:1601.00098](https://arxiv.org/abs/1601.00098)
2. M. Krstic, *Delay Compensation for Nonlinear, Adaptive, and PDE Systems*, Birkhauser, 2009.